In [ ]:
%pip install google-api-python-client pandas -q

In [ ]:
from googleapiclient.discovery import build
import pandas as pd
import time

# 🔑 Nhập API key của bạn
API_KEY = ""

# 🧑‍💻 ID của kênh YouTube (ví dụ: UC_x5XG1OV2P6uZZ5FSM9Ttw cho Google Developers)
CHANNEL_ID = "UCAuUUnT6oDeKwE6v1NGQxug"

In [ ]:
def get_uploads_playlist_id(youtube, channel_id):
    """Lấy playlist ID chứa toàn bộ video upload của kênh"""
    response = youtube.channels().list(
        part="contentDetails",
        id=channel_id
    ).execute()
    return response["items"][0]["contentDetails"]["relatedPlaylists"]["uploads"]

In [ ]:
def get_all_video_ids(youtube, playlist_id):
    """Lấy tất cả video_id từ playlist uploads"""
    video_ids = []
    next_page_token = None

    while True:
        request = youtube.playlistItems().list(
            part="contentDetails",
            playlistId=playlist_id,
            maxResults=50,
            pageToken=next_page_token
        )
        response = request.execute()

        for item in response["items"]:
            video_ids.append(item["contentDetails"]["videoId"])

        next_page_token = response.get("nextPageToken")
        if not next_page_token:
            break

    return video_ids

In [ ]:
import csv
import math

youtube = build("youtube", "v3", developerKey=API_KEY)

# B1: Lấy playlist chứa video của kênh
playlist_id = get_uploads_playlist_id(youtube, CHANNEL_ID)

# B2: Lấy toàn bộ video ID
video_ids = get_all_video_ids(youtube, playlist_id)
print(f"🔹 Tìm thấy {len(video_ids)} video trong kênh.")

# Số dòng mỗi file
rows_per_file = 500

# Tính số file cần tạo
num_files = math.ceil(len(video_ids) / rows_per_file)

for i in range(num_files):
    start = i * rows_per_file
    end = start + rows_per_file
    chunk = video_ids[start:end]

    filename = f"/content/drive/MyDrive/Colab Notebooks/HongTruc/Đồ án thực tế KHDL/links/video_ids_part_{i+1}.csv"
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["video_id"])  # tiêu đề cột
        for vid in chunk:
            writer.writerow([vid])

    print(f"✅ Đã lưu {len(chunk)} ID vào file: {filename}")

print("🎉 Hoàn tất chia nhỏ và lưu toàn bộ video IDs!")

🔹 Tìm thấy 5348 video trong kênh.
✅ Đã lưu 500 ID vào file: /content/drive/MyDrive/Colab Notebooks/HongTruc/Đồ án thực tế KHDL/video_ids_part_1.csv
✅ Đã lưu 500 ID vào file: /content/drive/MyDrive/Colab Notebooks/HongTruc/Đồ án thực tế KHDL/video_ids_part_2.csv
✅ Đã lưu 500 ID vào file: /content/drive/MyDrive/Colab Notebooks/HongTruc/Đồ án thực tế KHDL/video_ids_part_3.csv
✅ Đã lưu 500 ID vào file: /content/drive/MyDrive/Colab Notebooks/HongTruc/Đồ án thực tế KHDL/video_ids_part_4.csv
✅ Đã lưu 500 ID vào file: /content/drive/MyDrive/Colab Notebooks/HongTruc/Đồ án thực tế KHDL/video_ids_part_5.csv
✅ Đã lưu 500 ID vào file: /content/drive/MyDrive/Colab Notebooks/HongTruc/Đồ án thực tế KHDL/video_ids_part_6.csv
✅ Đã lưu 500 ID vào file: /content/drive/MyDrive/Colab Notebooks/HongTruc/Đồ án thực tế KHDL/video_ids_part_7.csv
✅ Đã lưu 500 ID vào file: /content/drive/MyDrive/Colab Notebooks/HongTruc/Đồ án thực tế KHDL/video_ids_part_8.cs

In [ ]:
import csv
import time
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# ------------------------------
# Cấu hình API
# ------------------------------
API_KEY = ""  # ⚠️ Thay bằng API key thật
youtube = build("youtube", "v3", developerKey=API_KEY)

# ------------------------------
# Hàm lấy bình luận từ 1 video
# ------------------------------
def get_comments_from_video(youtube, video_id):
    """Lấy bình luận từ 1 video"""
    comments = []
    next_page_token = None

    while True:
        try:
            request = youtube.commentThreads().list(
                part="snippet",
                videoId=video_id,
                maxResults=100,
                pageToken=next_page_token,
                textFormat="plainText"
            )
            response = request.execute()

            for item in response.get("items", []):
                snippet = item["snippet"]["topLevelComment"]["snippet"]
                comments.append({
                    "video_id": video_id,
                    "author": snippet.get("authorDisplayName", ""),
                    "comment": snippet.get("textDisplay", ""),
                    "published_at": snippet.get("publishedAt", ""),
                    "like_count": snippet.get("likeCount", 0)
                })

            next_page_token = response.get("nextPageToken")
            if not next_page_token:
                break

            # Tránh vượt tốc độ API
            time.sleep(0.3)

        except HttpError as e:
            if e.resp.status == 403:
                print(f"⚠️ Lỗi quota hoặc quyền truy cập (video {video_id}) — dừng lại.")
            else:
                print(f"❌ HttpError khi lấy bình luận video {video_id}: {e}")
            break
        except Exception as e:
            print(f"❌ Lỗi khi lấy bình luận video {video_id}: {e}")
            break

    return comments


# ------------------------------
# Hàm chính: đọc CSV đầu vào và lưu CSV kết quả
# ------------------------------
def collect_comments_from_csv(input_csv, output_csv, id_column="video_id"):
    all_comments = []

    # Đọc danh sách video_id từ file CSV
    with open(input_csv, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        video_ids = [row[id_column].strip() for row in reader if row.get(id_column)]

    print(f"📺 Đang lấy bình luận cho {len(video_ids)} video...")

    for i, video_id in enumerate(video_ids, start=1):
        print(f"\n🔹 [{i}/{len(video_ids)}] Video ID: {video_id}")
        comments = get_comments_from_video(youtube, video_id)
        print(f"   → Lấy được {len(comments)} bình luận.")
        all_comments.extend(comments)

        # Nghỉ một chút giữa các video
        time.sleep(0.5)

    # Ghi tất cả bình luận ra file CSV
    with open(output_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["video_id", "author", "comment", "published_at", "like_count"]
        )
        writer.writeheader()
        writer.writerows(all_comments)

    print(f"\n✅ Hoàn thành! Đã lưu {len(all_comments)} bình luận vào '{output_csv}'.")


# ------------------------------
# Chạy chương trình
# ------------------------------
if __name__ == "__main__":
    input_csv = "/content/drive/MyDrive/Colab Notebooks/HongTruc/Đồ án thực tế KHDL/links/video_ids_part_7.csv"   # 📥 File chứa danh sách video_id
    output_csv = "/content/drive/MyDrive/Colab Notebooks/HongTruc/Đồ án thực tế KHDL/data/comments_output_7.  csv"  # 💾 File kết quả
    collect_comments_from_csv(input_csv, output_csv)

📺 Đang lấy bình luận cho 500 video...

🔹 [1/500] Video ID: 7_RGn75JcZ8
   → Lấy được 227 bình luận.

🔹 [2/500] Video ID: 38OUCtzkT4Q
   → Lấy được 126 bình luận.

🔹 [3/500] Video ID: Bo8YN3oB0Cw
   → Lấy được 121 bình luận.

🔹 [4/500] Video ID: HR9956gDpUY
   → Lấy được 160 bình luận.

🔹 [5/500] Video ID: Zwwanld4T1w
   → Lấy được 199 bình luận.

🔹 [6/500] Video ID: ag33QJmknXM
   → Lấy được 158 bình luận.

🔹 [7/500] Video ID: I1cGiNnJZgU
   → Lấy được 40 bình luận.

🔹 [8/500] Video ID: akiQuyhXR8o
   → Lấy được 126 bình luận.

🔹 [9/500] Video ID: TeGr86rq06c
   → Lấy được 49 bình luận.

🔹 [10/500] Video ID: AYzA2uyd9_s
   → Lấy được 609 bình luận.

🔹 [11/500] Video ID: IfSkBBgF4a0
   → Lấy được 46 bình luận.

🔹 [12/500] Video ID: -vqV-gHa2FE
   → Lấy được 164 bình luận.



🔹 [13/500] Video ID: mWA2uL8zXPI
⚠️ Lỗi quota hoặc quyền truy cập (video mWA2uL8zXPI) — dừng lại.
   → Lấy được 0 bình luận.

🔹 [14/500] Video ID: ByEoeBYUwOo
   → Lấy được 71 bình luận.

🔹 [15/500] Video ID: zcMj4Az1MwE
   → Lấy được 590 bình luận.

🔹 [16/500] Video ID: esD6aaIjhek
   → Lấy được 156 bình luận.

🔹 [17/500] Video ID: N0vRIqeoefs
   → Lấy được 342 bình luận.

🔹 [18/500] Video ID: UG_X_7g63rY
   → Lấy được 319 bình luận.

🔹 [19/500] Video ID: qxmEpmCtM3o
   → Lấy được 159 bình luận.

🔹 [20/500] Video ID: n8yhaFd_GpM
   → Lấy được 1892 bình luận.

🔹 [21/500] Video ID: -GhVGZVcME8
   → Lấy được 171 bình luận.

🔹 [22/500] Video ID: xlUPlxSpDRo
   → Lấy được 44 bình luận.

🔹 [23/500] Video ID: JlbwchclCBo
   → Lấy được 52 bình luận.

🔹 [24/500] Video ID: es4w3WUcrN0
   → Lấy được 150 bình luận.

🔹 [25/500] Video ID: 7kkRkhAXZGg
   → Lấy được 585 bình luận.

🔹 [26/500] Video ID: aTfwA1TaH3Q
   → Lấy được 91 bình luận.

🔹 [27/500] Video ID: _pQ1BCdSTTI
   → Lấy được 39 bình lu


🔹 [30/500] Video ID: kTz52RW_bD0
⚠️ Lỗi quota hoặc quyền truy cập (video kTz52RW_bD0) — dừng lại.
   → Lấy được 0 bình luận.

🔹 [31/500] Video ID: z0HsPBKfhoI
   → Lấy được 717 bình luận.

🔹 [32/500] Video ID: 2ju61VPdTzw
   → Lấy được 93 bình luận.

🔹 [33/500] Video ID: w1R4F9s_oow
   → Lấy được 165 bình luận.

🔹 [34/500] Video ID: 8UPHFjHvGvY
   → Lấy được 496 bình luận.



🔹 [35/500] Video ID: ktOeFgmdIAo
⚠️ Lỗi quota hoặc quyền truy cập (video ktOeFgmdIAo) — dừng lại.
   → Lấy được 0 bình luận.



🔹 [36/500] Video ID: dY9f9bFctUE
⚠️ Lỗi quota hoặc quyền truy cập (video dY9f9bFctUE) — dừng lại.
   → Lấy được 0 bình luận.

🔹 [37/500] Video ID: bVV2Zk88beY
   → Lấy được 9332 bình luận.

🔹 [38/500] Video ID: 6cL5Nud8d7w
   → Lấy được 592 bình luận.

🔹 [39/500] Video ID: HfMhU6qjVVM
   → Lấy được 68 bình luận.

🔹 [40/500] Video ID: aR5N2Jl8k14
   → Lấy được 5412 bình luận.


...

⚠️ Lỗi quota hoặc quyền truy cập (video UxLRv0FEndM) — dừng lại.
   → Lấy được 100 bình luận.



🔹 [497/500] Video ID: rSQNi5sAwuc
⚠️ Lỗi quota hoặc quyền truy cập (video rSQNi5sAwuc) — dừng lại.
   → Lấy được 0 bình luận.



🔹 [498/500] Video ID: MnT1xgZgkpk
⚠️ Lỗi quota hoặc quyền truy cập (video MnT1xgZgkpk) — dừng lại.
   → Lấy được 0 bình luận.



🔹 [499/500] Video ID: fTsFVO6OhO8
⚠️ Lỗi quota hoặc quyền truy cập (video fTsFVO6OhO8) — dừng lại.
   → Lấy được 0 bình luận.

🔹 [500/500] Video ID: Us70DN2XSfM


⚠️ Lỗi quota hoặc quyền truy cập (video Us70DN2XSfM) — dừng lại.
   → Lấy được 100 bình luận.

✅ Hoàn thành! Đã lưu 348678 bình luận vào '/content/drive/MyDrive/Colab Notebooks/HongTruc/Đồ án thực tế KHDL/data/comments_output_7.  csv'.
